# I. Imports & Chargement modèles

In [3]:
%pip install roboflow

import streamlit as st
from roboflow import Roboflow
import cv2
import numpy as np
import tempfile
from tqdm import tqdm

st.title("Football 2D Video Analyzer")
st.write("Analyse de matchs et génération de radar/2D vidéo.")

api_key = "oetSR0O32rWFikQEeiNw"

def get_model(model_id, api_key):
    rf = Roboflow(api_key=api_key)
    project_name, version = model_id.split('/')
    project = rf.workspace().project(project_name)
    model = project.version(version).model
    return model

@st.cache_resource
def load_models(api_key):
    PLAYER_DETECTION_MODEL_ID = "football-players-detection-3zvbc/10"
    FIELD_DETECTION_MODEL_ID = "football-field-detection-f07vi/15"
    player_model = get_model(model_id=PLAYER_DETECTION_MODEL_ID, api_key=api_key)
    field_model = get_model(model_id=FIELD_DETECTION_MODEL_ID, api_key=api_key)
    return player_model, field_model

if api_key:
    PLAYER_DETECTION_MODEL, FIELD_DETECTION_MODEL = load_models(api_key)
else:
    PLAYER_DETECTION_MODEL, FIELD_DETECTION_MODEL = None, None



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\moham\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
2025-07-14 21:50:32.163 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-14 21:50:32.164 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-14 21:50:32.164 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-14 21:50:32.165 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-14 21:50:32.166 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-14 21:50:32.166 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-14 21:50:

Note: you may need to restart the kernel to use updated packages.
loading Roboflow workspace...


2025-07-14 21:50:32.676 Thread 'Thread-11': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-14 21:50:32.677 Thread 'Thread-11': missing ScriptRunContext! This warning can be ignored when running in bare mode.


loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...


2025-07-14 21:50:38.361 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-14 21:50:38.362 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


# II. Upload Vidéo


In [4]:
!gdown -O "0bfacc_0.mp4" "https://drive.google.com/uc?id=12TqauVZ9tLAv8kWxTTBFWtgt2hNQ4_ZF"
!gdown -O "2e57b9_0.mp4" "https://drive.google.com/uc?id=19PGw55V8aA6GZu5-Aac5_9mCy3fNxmEf"
!gdown -O "08fd33_0.mp4" "https://drive.google.com/uc?id=1OG8K6wqUw9t7lp9ms1M48DxRhwTYciK-"
!gdown -O "573e61_0.mp4" "https://drive.google.com/uc?id=1yYPKuXbHsCxqjA9G-S6aeR2Kcnos8RPU"
!gdown -O "121364_0.mp4" "https://drive.google.com/uc?id=1vVwjW1dE1drIdd4ZSILfbCGPD4weoNiu"

Downloading...
From: https://drive.google.com/uc?id=12TqauVZ9tLAv8kWxTTBFWtgt2hNQ4_ZF
To: c:\Users\moham\Documents\Github\HGP-GO\sports\examples\soccer\notebooks\0bfacc_0.mp4

  0%|          | 0.00/19.9M [00:00<?, ?B/s]
  5%|▌         | 1.05M/19.9M [00:00<00:02, 8.23MB/s]
 11%|█         | 2.10M/19.9M [00:00<00:02, 8.15MB/s]
 16%|█▌        | 3.15M/19.9M [00:00<00:02, 8.11MB/s]
 21%|██        | 4.19M/19.9M [00:00<00:01, 8.69MB/s]
 26%|██▋       | 5.24M/19.9M [00:00<00:01, 8.52MB/s]
 32%|███▏      | 6.29M/19.9M [00:00<00:01, 8.65MB/s]
 37%|███▋      | 7.34M/19.9M [00:00<00:01, 8.75MB/s]
 42%|████▏     | 8.39M/19.9M [00:00<00:01, 8.89MB/s]
 47%|████▋     | 9.44M/19.9M [00:01<00:01, 8.97MB/s]
 53%|█████▎    | 10.5M/19.9M [00:01<00:01, 9.19MB/s]
 58%|█████▊    | 11.5M/19.9M [00:01<00:00, 8.98MB/s]
 63%|██████▎   | 12.6M/19.9M [00:01<00:00, 8.97MB/s]
 69%|██████▊   | 13.6M/19.9M [00:01<00:00, 8.97MB/s]
 74%|███████▍  | 14.7M/19.9M [00:01<00:00, 9.08MB/s]
 79%|███████▉  | 15.7M/19.9M [00:01<00

In [5]:
uploaded_file = st.file_uploader("Sélectionner la vidéo", type=["mp4", "avi"])
if uploaded_file:
    with tempfile.NamedTemporaryFile(delete=False, suffix='.mp4') as tmp_input:
        tmp_input.write(uploaded_file.read())
        input_path = tmp_input.name

    st.video(input_path)


2025-07-14 21:52:25.616 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-14 21:52:25.616 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-14 21:52:25.616 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-14 21:52:25.617 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-14 21:52:25.617 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


# III. Pipeline Frame Unique (Preview d’une Frame)

In [6]:
if uploaded_file and PLAYER_DETECTION_MODEL:
    # Lire une frame pour test
    cap = cv2.VideoCapture(input_path)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        st.error("Impossible de lire la vidéo.")
    else:
        result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
        detections = sv.Detections.from_inference(result)
        
        # Annoter avec Box et Label (exemple)
        box_annotator = sv.BoxAnnotator(
            color=sv.ColorPalette.from_hex(['#FF8C00', '#00BFFF', '#FF1493', '#FFD700']),
            thickness=2
        )
        label_annotator = sv.LabelAnnotator(
            color=sv.ColorPalette.from_hex(['#FF8C00', '#00BFFF', '#FF1493', '#FFD700']),
            text_color=sv.Color.from_hex('#000000')
        )
        labels = [
            f"{class_name} {confidence:.2f}"
            for class_name, confidence
            in zip(detections['class_name'], detections.confidence)
        ]

        annotated_frame = box_annotator.annotate(
            scene=frame.copy(),
            detections=detections)
        annotated_frame = label_annotator.annotate(
            scene=annotated_frame,
            detections=detections,
            labels=labels)
        
        st.image(annotated_frame, channels="BGR", caption="Frame annotée")


# IV. Génération vidéo 2D complète

In [7]:
def process_video(input_path, output_path, player_model, team_classifier=None):
    # Reprends ici toute ta logique (détection, tracking, team, projection, etc.)
    cap = cv2.VideoCapture(input_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    pbar = tqdm(total=total_frames)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # Pipeline complet sur chaque frame (copie tes blocs ici)
        # result = player_model.infer(frame, confidence=0.3)[0]
        # detections = sv.Detections.from_inference(result)
        # ... etc.
        out.write(frame)  # Remplace par frame annotée
        pbar.update(1)
    pbar.close()
    cap.release()
    out.release()
    return output_path

if uploaded_file and PLAYER_DETECTION_MODEL:
    if st.button("Générer vidéo 2D"):
        output_path = input_path.replace('.mp4', '_2d.mp4')
        with st.spinner("Traitement vidéo..."):
            process_video(input_path, output_path, PLAYER_DETECTION_MODEL)
            st.video(output_path)
            with open(output_path, "rb") as f:
                st.download_button("Télécharger vidéo 2D", f, "result_2d.mp4", "video/mp4")


In [11]:
!pip install -q git+https://github.com/roboflow/sports.git


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\moham\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
import cv2
import numpy as np
import supervision as sv
from tqdm import tqdm
from sports.annotators.soccer import (
    draw_pitch,
    draw_points_on_pitch,
    draw_pitch_voronoi_diagram
)
from sports.common.view import ViewTransformer
from sports.configs.soccer import SoccerPitchConfiguration

def process_video(
    input_path, 
    output_path, 
    player_model, 
    field_model,
    team_classifier,  # déjà fit en amont !
    config=None,
    stride=1,
    progress_bar=True
):
    if config is None:
        config = SoccerPitchConfiguration()

    # IDs classes
    BALL_ID = 0
    GOALKEEPER_ID = 1
    PLAYER_ID = 2
    REFEREE_ID = 3

    # Annotateurs
    ellipse_annotator = sv.EllipseAnnotator(
        color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
        thickness=2
    )
    label_annotator = sv.LabelAnnotator(
        color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
        text_color=sv.Color.from_hex('#000000'),
        text_position=sv.Position.BOTTOM_CENTER
    )
    triangle_annotator = sv.TriangleAnnotator(
        color=sv.Color.from_hex('#FFD700'),
        base=20, height=17
    )
    tracker = sv.ByteTrack()
    tracker.reset()

    # Lecture vidéo
    cap = cv2.VideoCapture(input_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    pbar = tqdm(total=total_frames, disable=not progress_bar)
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1
        if stride > 1 and frame_idx % stride != 0:
            continue

        # Détection objets
        result = player_model.infer(frame, confidence=0.3)[0]
        detections = sv.Detections.from_inference(result)

        ball_detections = detections[detections.class_id == BALL_ID]
        ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

        all_detections = detections[detections.class_id != BALL_ID]
        all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
        all_detections = tracker.update_with_detections(detections=all_detections)

        goalkeepers_detections = all_detections[all_detections.class_id == GOALKEEPER_ID]
        players_detections = all_detections[all_detections.class_id == PLAYER_ID]
        referees_detections = all_detections[all_detections.class_id == REFEREE_ID]

        # Assignment équipes
        players_crops = [sv.crop_image(frame, xyxy) for xyxy in players_detections.xyxy]
        players_detections.class_id = team_classifier.predict(players_crops)
        # À adapter selon ta logique d'init du team_classifier (fit avant sur N frames)

        goalkeepers_detections.class_id = resolve_goalkeepers_team_id(
            players_detections, goalkeepers_detections
        )
        referees_detections.class_id -= 1

        all_detections = sv.Detections.merge([
            players_detections, goalkeepers_detections, referees_detections])

        # Frame radar 2D
        # Détection du terrain et projection
        result_field = field_model.infer(frame, confidence=0.3)[0]
        key_points = sv.KeyPoints.from_inference(result_field)
        filter_kp = key_points.confidence[0] > 0.5
        frame_reference_points = key_points.xy[0][filter_kp]
        pitch_reference_points = np.array(config.vertices)[filter_kp]

        if len(frame_reference_points) < 4:  # Pas assez de points, skip
            pbar.update(1)
            continue

        transformer = ViewTransformer(
            source=frame_reference_points,
            target=pitch_reference_points
        )

        frame_ball_xy = ball_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        pitch_ball_xy = transformer.transform_points(points=frame_ball_xy)

        players_xy = players_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        pitch_players_xy = transformer.transform_points(points=players_xy)

        referees_xy = referees_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        pitch_referees_xy = transformer.transform_points(points=referees_xy)

        # Dessin radar 2D
        annotated_frame = draw_pitch(config)
        annotated_frame = draw_points_on_pitch(
            config=config,
            xy=pitch_ball_xy,
            face_color=sv.Color.WHITE,
            edge_color=sv.Color.BLACK,
            radius=10,
            pitch=annotated_frame)
        annotated_frame = draw_points_on_pitch(
            config=config,
            xy=pitch_players_xy[players_detections.class_id == 0],
            face_color=sv.Color.from_hex('00BFFF'),
            edge_color=sv.Color.BLACK,
            radius=16,
            pitch=annotated_frame)
        annotated_frame = draw_points_on_pitch(
            config=config,
            xy=pitch_players_xy[players_detections.class_id == 1],
            face_color=sv.Color.from_hex('FF1493'),
            edge_color=sv.Color.BLACK,
            radius=16,
            pitch=annotated_frame)
        annotated_frame = draw_points_on_pitch(
            config=config,
            xy=pitch_referees_xy,
            face_color=sv.Color.from_hex('FFD700'),
            edge_color=sv.Color.BLACK,
            radius=16,
            pitch=annotated_frame)

        # Ajoute ici voronoi ou autres visualisations si besoin

        out.write(annotated_frame)
        pbar.update(1)

    pbar.close()
    cap.release()
    out.release()
    return output_path
